In [3]:
import os
import time
import shutil
import numpy as np
from glob import glob
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import optimizers

from config import IMG_SIZE, BATCH_SIZE, LEARNING_RATE
from config import NEW_DINO_FOLDER, NEW_NON_DINO_FOLDER
from config import MODEL_PATH, DINO_FOLDER, NON_DINO_FOLDER

In [2]:
def load_old_dataset():
    def load(folder, label):
        images, labels = [], []
        for path in glob(os.path.join(folder, "**/*.*"), recursive=True):
            try:
                img = Image.open(path).convert("RGB").resize(IMG_SIZE)
                images.append(np.array(img))
                labels.append(label)
            except:
                pass
        return np.array(images), np.array(labels)

    dino_imgs, dino_lbls = load(DINO_FOLDER, 1)
    nondino_imgs, nondino_lbls = load(NON_DINO_FOLDER, 0)

    X_old = np.concatenate([dino_imgs, nondino_imgs], axis=0).astype("float32") / 255.0
    y_old = np.concatenate([dino_lbls, nondino_lbls], axis=0)

    print(f"Старий датасет завантажено: {X_old.shape}")
    return X_old, y_old

In [3]:
def load_images_from_folder(folder, label):
    images, labels = [], []
    for path in glob(os.path.join(folder, "**/*.*"), recursive=True):
        try:
            img = Image.open(path).convert("RGB").resize(IMG_SIZE)
            images.append(np.array(img))
            labels.append(label)
        except:
            print(f"Пропускаємо файл: {path}")
    return np.array(images), np.array(labels)

In [4]:
new_dino_images, new_dino_labels = load_images_from_folder(NEW_DINO_FOLDER, 1)
new_non_dino_images, new_non_dino_labels = load_images_from_folder(NEW_NON_DINO_FOLDER, 0)

X_new = np.concatenate([new_dino_images, new_non_dino_images], axis=0).astype("float32") / 255.0
y_new = np.concatenate([new_dino_labels, new_non_dino_labels], axis=0)

In [5]:
if len(X_new) == 0:
    print("❗ Немає нових даних для донавчання.")
    exit()

print(f"Нових даних: {X_new.shape}")

Нових даних: (5997, 150, 150, 3)


In [6]:
X_old, y_old = load_old_dataset()

Старий датасет завантажено: (9196, 150, 150, 3)


In [7]:
X_full = np.concatenate([X_old, X_new], axis=0)
y_full = np.concatenate([y_old, y_new], axis=0)

print(f"Повний датасет для донавчання: {X_full.shape}")

Повний датасет для донавчання: (15193, 150, 150, 3)


In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

In [9]:
model = load_model(MODEL_PATH)
print("Модель завантажена!")

Модель завантажена!


In [10]:
for layer in model.layers[:-2]:
    layer.trainable = False

In [11]:
train_gen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
)
val_gen = ImageDataGenerator()

train_flow = train_gen.flow(X_train, y_train, batch_size=BATCH_SIZE)
val_flow = val_gen.flow(X_val, y_val, batch_size=BATCH_SIZE)

In [12]:
classes = np.unique(y_train)
class_weights_array = compute_class_weight("balanced", classes=classes, y=y_train)
class_weights = {cls: w for cls, w in zip(classes, class_weights_array)}


In [13]:
model.compile(
    loss="binary_crossentropy",
    optimizer=optimizers.Adam(LEARNING_RATE),
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    ModelCheckpoint("fine_tuned_model.keras", save_best_only=True)
]

model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=2
)

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
380/380 - 110s - 289ms/step - accuracy: 0.8404 - loss: 0.3694 - val_accuracy: 0.8756 - val_loss: 0.3161 - learning_rate: 1.0000e-04
Epoch 2/15
380/380 - 103s - 270ms/step - accuracy: 0.8532 - loss: 0.3410 - val_accuracy: 0.8812 - val_loss: 0.3071 - learning_rate: 1.0000e-04
Epoch 3/15
380/380 - 111s - 292ms/step - accuracy: 0.8570 - loss: 0.3331 - val_accuracy: 0.8829 - val_loss: 0.3016 - learning_rate: 1.0000e-04
Epoch 4/15
380/380 - 106s - 278ms/step - accuracy: 0.8602 - loss: 0.3251 - val_accuracy: 0.8838 - val_loss: 0.2978 - learning_rate: 1.0000e-04
Epoch 5/15
380/380 - 107s - 281ms/step - accuracy: 0.8647 - loss: 0.3169 - val_accuracy: 0.8881 - val_loss: 0.2860 - learning_rate: 1.0000e-04
Epoch 6/15
380/380 - 106s - 279ms/step - accuracy: 0.8644 - loss: 0.3124 - val_accuracy: 0.8917 - val_loss: 0.2829 - learning_rate: 1.0000e-04
Epoch 7/15
380/380 - 106s - 279ms/step - accuracy: 0.8658 - loss: 0.3101 - val_accuracy: 0.8881 - val_loss: 0.2863 - learning_rate: 1.0000e-04

In [14]:
model.save("test_new_model.h5")
print(f"Модель збережена у: test_new_model.h5")

Модель збережена у: test_new_model.h5


In [ ]:
model.save(MODEL_PATH)
print(f"Модель збережена у: {MODEL_PATH}")

In [1]:
import os
import shutil
import time
import random

def move_new_data_with_subfolders(src_folders, dst_base_folders):
    """
    Переміщує всі файли з підпапок src_folders у відповідні dst_base_folders.
    Якщо підпапка вже існує у dst, створюється нова з унікальною назвою.
    """
    for src, dst_base in zip(src_folders, dst_base_folders):
        for root, dirs, files in os.walk(src):
            # Відносний шлях підпапки від src
            rel_path = os.path.relpath(root, src)
            if rel_path == ".":
                rel_path = ""
            
            # Цільова підпапка
            dst_subfolder = os.path.join(dst_base, rel_path)
            
            # Якщо така папка вже існує, додаємо унікальний суфікс
            base_dst_subfolder = dst_subfolder
            counter = 1
            while os.path.exists(dst_subfolder):
                dst_subfolder = f"{base_dst_subfolder}_new{counter}"
                counter += 1
            os.makedirs(dst_subfolder, exist_ok=True)
            
            # Переміщуємо файли у цільову підпапку
            for file in files:
                src_path = os.path.join(root, file)
                filename, ext = os.path.splitext(file)
                dst_path = os.path.join(dst_subfolder, file)
                
                # Якщо файл з таким ім’ям уже існує, додаємо унікальний суфікс
                while os.path.exists(dst_path):
                    rand_suffix = f"_new{int(time.time()*1000)%100000}_{random.randint(0,999)}"
                    dst_path = os.path.join(dst_subfolder, filename + rand_suffix + ext)
                
                shutil.move(src_path, dst_path)
        
        # Створюємо порожню папку src для наступного завантаження
        os.makedirs(src, exist_ok=True)

        print(f"Файли з {src} переміщено у {dst_base} з підпапками та унікальними назвами.")


In [4]:
move_new_data_with_subfolders(
    src_folders=[NEW_DINO_FOLDER, NEW_NON_DINO_FOLDER],
    dst_base_folders=[DINO_FOLDER, NON_DINO_FOLDER]
)

Файли з dataset/new_dino переміщено у dataset/dinosaur з підпапками та унікальними назвами.
Файли з dataset/new_non_dino переміщено у dataset/not_dinosaur з підпапками та унікальними назвами.


In [24]:
def clear_subfolders(folder):
    if not os.path.exists(folder):
        return

    for item in os.listdir(folder):
        item_path = os.path.join(folder, item)

        try:
            if os.path.isdir(item_path):
                shutil.rmtree(item_path)
            else:
                os.remove(item_path)
        except PermissionError:
            print(f"Файл зайнятий, повторюю через 0.5 секунди: {item_path}")
            time.sleep(0.5)
            try:
                if os.path.isdir(item_path):
                    shutil.rmtree(item_path)
                else:
                    os.remove(item_path)
            except:
                print(f"Не можу видалити файл: {item_path}")

clear_subfolders(NEW_DINO_FOLDER)
clear_subfolders(NEW_NON_DINO_FOLDER)

print("Папки очищено. Лишилися лише порожні NEW_DINO і NEW_NON_DINO")

Папки очищено. Лишилися лише порожні NEW_DINO і NEW_NON_DINO
